# Import neccessary libraries and dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve, RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import RFECV, RFE
from sklearn.metrics import make_scorer, f1_score, confusion_matrix, classification_report, accuracy_score, roc_curve, auc, roc_auc_score, precision_recall_curve
import time
import os
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from collections import OrderedDict
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
df = pd.read_csv('features_dataset.csv')
df.head()

In [ ]:
df = df.drop(columns=["file", "row", "col", "object", "iteration"], errors="ignore")
df.describe().T

In [ ]:
X = df.drop(columns=['presence'])
y = df['presence'].astype(int)

# 1. Tree/Simple Neural Network models

## 1.1 Feature Selection

### Step 1: Remove Highly Correlated Features (Collinearity)

#### Train/test split and normalization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle = True, stratify=y, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

In [ ]:
print('Train shape: ' + str(X_train.shape))
print('Test shape: ' + str(X_test.shape))

#### Correlation filtering

In [ ]:
target_corr = X_train_scaled_df.corrwith(y).abs()
corr_matrix = X_train_scaled_df.corr().abs()
to_drop = set()

for i in range(len(corr_matrix.columns)):
        for j in range(i):
            # If features i and j are highly correlated
            if corr_matrix.iloc[i, j] > 0.95:
                colname_i = corr_matrix.columns[i]
                colname_j = corr_matrix.columns[j]

                # COMPARE: Which one correlates more with Y
                # If i is weaker than j, drop i.
                if target_corr[colname_i] < target_corr[colname_j]:
                    to_drop.add(colname_i)
                else:
                    to_drop.add(colname_j)

plt.figure(figsize = (45, 45))
sns.heatmap(corr_matrix, annot = True)

print(target_corr)
X_train_reduced = X_train_scaled_df.drop(columns=to_drop)

In [ ]:
print(f"Smart selection dropping {len(to_drop)} features: {list(to_drop)}")
print('\n')
print('Shape of the remaining dataset after step 1: ' + (str(X_train_reduced.shape)))

In [ ]:
X_train_reduced

### Step 2: Tree-Based Feature Importance (Non-Linear Check)

#### Feature importance ranking

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_reduced, y_train)

importances = pd.DataFrame({
    'Feature': X_train_reduced.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Plot Top Features
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"] + plt.rcParams["font.serif"]
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 8
plt.rcParams["ytick.labelsize"] = 8

plt.figure(figsize=(3.5, 4))
sns.barplot(
    x='Importance', 
    y='Feature', 
    data=importances, 
    hue='Feature',      # Explicitly map the y-variable to the color palette
    palette='viridis', 
    legend=False        # Suppress the redundant legend
)
plt.xlabel('Importance Score')
plt.ylabel('Features')
sns.despine()
plt.tight_layout()
plt.savefig('feature_importance.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

### Step 3: Recursive Feature Elimination (RFECV)

In [ ]:
f1_scorer = make_scorer(f1_score, average='weighted')

In [ ]:
f1_scorer = make_scorer(f1_score, average='weighted')
rfecv = RFECV(
    estimator=rf,
    step=1,              # Remove 1 feature at each iteration
    cv=5,                # Use 5-fold cross-validation
    scoring=f1_scorer,  # Use f1 score to determine the best subset
    min_features_to_select=5 # Keep at least 5 features
)

rfecv.fit(X_train_reduced, y_train)
optimal_features_count = rfecv.n_features_
print(f"Optimal number of features: {optimal_features_count}")

In [ ]:
min_features = 5 
num_scores = len(rfecv.cv_results_['mean_test_score'])
x_axis_features = range(min_features, min_features + num_scores)

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"] + plt.rcParams["font.serif"]
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 8
plt.rcParams["ytick.labelsize"] = 8

plt.figure(figsize=(3.5, 3))

plt.plot(x_axis_features, rfecv.cv_results_['mean_test_score'], 
         linestyle='--', marker='o', markersize=3, 
         color='black', linewidth=1, label='CV Score')

plt.xlabel('Number of Features Selected')
plt.ylabel('F1 Score (Mean CV)')

if len(x_axis_features) > 10:
    plt.xticks(np.arange(min_features, min_features + num_scores, 2))
else:
    plt.xticks(np.arange(min_features, min_features + num_scores, 1))

plt.grid(axis='y', linestyle='--', alpha=0.7)
sns.despine() # If you have seaborn imported, otherwise use plt.gca().spines[['top', 'right']].set_visible(False)

plt.tight_layout()

plt.savefig('rfecv_plot2.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
selected_mask = rfecv.support_
final_features = X_train_reduced.columns[selected_mask]
print(f"Total features kept: {len(final_features)}")
print(final_features.tolist())

X_train_final = X_train_scaled_df[final_features]
X_test_final = X_test_scaled_df[final_features]

## 1.2 Random Forest

### Hyperparameter tuning

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None], # None means nodes are expanded until all leaves are pure
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced', None]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

In [ ]:
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=5, # Use 5-fold cross-validation
    verbose=2, # Print status updates
    n_jobs=-1 # Use all available processors
)

print("Starting Grid Search...")
start_time = time.time()

grid_search.fit(X_train_final, y_train)

end_time = time.time()
print(f"Grid Search Complete. Time taken: {round((end_time - start_time)/60, 2)} minutes.")

In [ ]:
best_params = grid_search.best_params_
print("\n--- Best Hyperparameters Found ---")
print(best_params)

best_rf = grid_search.best_estimator_
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")

y_pred = best_rf.predict(X_test_final)
final_f1_score = f1_score(y_test, y_pred, average='weighted')
print(f"\n--- Final Performance on Test Set ---")
print(f"Optimized Random Forest F1 Score: {final_f1_score:.4f}")

### Performance Evaluation

#### Confusion matrix and classification report

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("--- CLASSIFICATION REPORT (Optimized RF) ---")
print(classification_report(y_test, y_pred, target_names=['No Object (0)', 'Foreign Object (1)']))

final_f1_score = f1_score(y_test, y_pred, average='weighted')
print(f"Final Weighted F1 Score: {final_f1_score:.4f}")


class_names = ['No Object', 'Foreign Object']
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"] + plt.rcParams["font.serif"]
plt.rcParams["font.size"] = 10
plt.figure(figsize=(3.5, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            annot_kws={"size": 9}, 
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix2.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

#### Check Overfitting

In [ ]:
y_train_pred = best_rf.predict(X_train_final)
train_f1 = f1_score(y_train, y_train_pred, average='weighted')
test_f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Training F1 Score: {train_f1:.4f}")
print(f"Test F1 Score:     {test_f1:.4f}")

score_gap = train_f1 - test_f1
print(f"Score Gap:         {score_gap:.4f}")

##### Learning curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_learning_curve_ieee(estimator, X, y):
    from sklearn.model_selection import learning_curve
    
    # 1. IEEE Global Styling
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["Times New Roman"] + plt.rcParams["font.serif"]
    plt.rcParams["font.size"] = 10

    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=5, n_jobs=-1, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='f1_weighted'
    )

    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)

    # 2. IEEE Sizing (Single Column)
    plt.figure(figsize=(3.5, 3)) 

    # 3. Plotting with high-contrast colors and distinct markers
    # Using Blue and Dark Orange (more accessible than Red/Green)
    plt.plot(train_sizes, train_scores_mean, 's-', color="#1f77b4", 
             markersize=3, linewidth=1, label="Training Score")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="#ff7f0e", 
             markersize=3, linewidth=1, label="Cross-validation Score")

    # 4. Subtle standard deviation bands
    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.15, color="#1f77b4")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.15, color="#ff7f0e")

    # 5. Labels and Legend
    plt.xlabel("Training Examples")
    plt.ylabel("F1 Score (Weighted)") # Be specific about the score
    plt.legend(loc="lower right", fontsize=8, frameon=True) # Legend inside, smaller font
    
    # 6. Cleaning the grid and spines
    plt.grid(linestyle='--', alpha=0.6)
    plt.gca().spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('learning_curve.pdf', format='pdf', dpi=300, bbox_inches='tight')
    plt.show()

# Run the function
plot_learning_curve_ieee(best_rf, X_train_final, y_train)

 ##### Repeated Evaluation for Stability

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

scores = cross_val_score(best_rf, X_train_final, y_train, scoring='f1_weighted', cv=cv, n_jobs=-1)

# Report stability
mean_score = np.mean(scores)
std_dev = np.std(scores)
print(f"Weighted F1 Score: {mean_score:.4f} (Standard Deviation: {std_dev:.4f})")
print(f"Coefficient of Variation (CV): {(std_dev / mean_score * 100):.2f}%")

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"] + plt.rcParams["font.serif"]
plt.rcParams["font.size"] = 10

# 2. IEEE Sizing (Single Column)
plt.figure(figsize=(3.5, 2)) 

# 3. Enhanced Visualization
# Combine a boxplot with a stripplot to show individual trial results
sns.boxplot(x=scores, color='#e6f2ff', width=0.4, linewidth=1.2, showfliers=False)
# sns.stripplot(x=scores, color='#1f77b4', size=3, alpha=0.6, jitter=True)

plt.xlabel('Weighted F1 Score')
plt.yticks([]) # Hide Y-axis as it's a single distribution

# 5. Clean up the grid and spines
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.gca().spines[['top', 'right', 'left']].set_visible(False)

# 6. Optional: Annotate the Mean/Median for clarity
mean_val = np.mean(scores)
plt.text(mean_val, 0.35, f'Mean: {mean_val:.3f}', 
         ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()

# 7. Save as Vector PDF
plt.savefig('model_stability2.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import feature_engineering as fe
import glob
from pathlib import Path
from tqdm import tqdm

In [ ]:
TEST_FOLDER = 'test'

def process_new_test_files():
    records = []

    files = glob.glob(f"{TEST_FOLDER}/*.csv") 

    if not files:
        print(f"No CSV files found in the '{TEST_FOLDER}' directory.")
        return pd.DataFrame()

    for fname in tqdm(files, desc="Processing Test Files"):
        # Labels might be unnecessary for test prediction, but we use the file name
        labels = {'file': Path(fname).name} 

        try:
            # Replicate original data loading logic
            df = pd.read_csv(fname, skiprows=4, usecols=['Time', 'Ampl'])

            if df.empty or 'Time' not in df or 'Ampl' not in df:
                print(f"Skipping empty or invalid file: {Path(fname).name}")
                continue

            t = df['Time'].to_numpy()
            x = df['Ampl'].to_numpy()
            
            # Use the imported feature extraction function
            feats = fe.compute_features_from_array(t, x)
            rec = {**labels, **feats}
            records.append(rec)

        except pd.errors.EmptyDataError:
            print(f"Skipping empty file: {Path(fname).name}")
            continue
        except Exception as e:
            print(f"Error processing {Path(fname).name}: {e}")
            continue

    return pd.DataFrame(records)

In [ ]:
X_new_raw = process_new_test_files()

In [ ]:
FINAL_FEATURES = final_features.tolist()
SCALER = scaler
MODEL = best_rf
SCALER_FIT_COLUMNS = X_train.columns.tolist()

X_new_for_scaling = X_new_raw[SCALER_FIT_COLUMNS]

X_new_scaled_np = SCALER.transform(X_new_for_scaling)
X_new_scaled_df = pd.DataFrame(X_new_scaled_np, columns=SCALER_FIT_COLUMNS)
X_new_final = X_new_scaled_df[FINAL_FEATURES]

In [ ]:
X_new_final

In [ ]:
# 4. Generate Predictions
new_data_predictions = MODEL.predict(X_new_final)
new_data_probabilities = MODEL.predict_proba(X_new_final)[:, 1]

# 5. Create Final Report
prediction_report = pd.DataFrame({
    'File_Name': X_new_raw['file'],
    'Prediction_0_1': new_data_predictions,
    'Confidence_Score (Class 1)': new_data_probabilities
})

print("\n--- UNSEEN FILE PREDICTION REPORT ---")
print(prediction_report)

## 1.3 XGBoost (Extreme Gradient Boosting)

### Hyperparameter tuning

In [ ]:
num_negative = (y_train == 0).sum()
num_positive = (y_train == 1).sum()
scale_ratio = num_negative / num_positive

print(f"Calculated scale_pos_weight: {scale_ratio:.2f}")

xgb_estimator = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],         # Train on 80% or 100% of data per tree
    'colsample_bytree': [0.8, 1.0],   # Train on 80% or 100% of features per tree
    'scale_pos_weight': [1, scale_ratio]
}

xgb_grid = GridSearchCV(
    estimator=xgb_estimator,
    param_grid=xgb_param_grid,
    scoring=f1_scorer,
    cv=5,
    verbose=1,
    n_jobs=-1
)

print("Starting XGBoost Hyperparameter Tuning...")
start_time = time.time()
xgb_grid.fit(X_train_final, y_train)
print(f"XGBoost Tuning Complete. Time: {round((time.time() - start_time)/60, 2)} min.")

best_xgb = xgb_grid.best_estimator_
print(f"Best XGBoost Params: {xgb_grid.best_params_}")
print(f"Best CV Score: {xgb_grid.best_score_:.4f}")

In [ ]:
y_pred = best_xgb.predict(X_test_final)
final_f1_score_xgb = f1_score(y_test, y_pred, average='weighted')
print(f"\n--- Final Performance on Test Set ---")
print(f"Optimized XGBoost F1 Score: {final_f1_score_xgb:.4f}")

## 1.4 Neural Network

### Hyperparameter tuning

In [ ]:
mlp_estimator = MLPClassifier(
    max_iter=1000, # Increased to ensure convergence
    random_state=42
)

mlp_param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (50, 25), (100, 50)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],  # Regularization strength
    'learning_rate_init': [0.001, 0.01] # Initial learning rate
}

mlp_grid = GridSearchCV(
    estimator=mlp_estimator,
    param_grid=mlp_param_grid,
    scoring=f1_scorer,
    cv=5,
    verbose=1,
    n_jobs=-1
)

print("\nStarting Neural Network Tuning...")
start_time = time.time()
mlp_grid.fit(X_train_final, y_train)
print(f"MLP Tuning Complete. Time: {round((time.time() - start_time)/60, 2)} min.")

# 4. Get Best Results
best_mlp = mlp_grid.best_estimator_
print(f"Best MLP Params: {mlp_grid.best_params_}")
print(f"Best CV Score: {mlp_grid.best_score_:.4f}")

In [ ]:
y_pred = best_mlp.predict(X_test_final)
final_f1_score_mlp = f1_score(y_test, y_pred, average='weighted')
print(f"\n--- Final Performance on Test Set ---")
print(f"Optimized MLP Score: {final_f1_score_mlp:.4f}")

# 2. Linear Model

## 2.1 Feature Selection

In [ ]:
C_values = np.logspace(-4, 4, 10)

lasso_selector = LogisticRegressionCV(
    Cs=C_values, 
    penalty='l1', 
    solver='liblinear', # Required solver for L1 penalty
    cv=5, 
    scoring='f1_weighted', # Optimize for F1 score
    random_state=42, 
    n_jobs=-1,
    max_iter = 500
)

print("Running Lasso Feature Selection for Logistic Regression...")
lasso_selector.fit(X_train_reduced, y_train)

In [ ]:
# The coefficients are stored in the first array index [0] for binary classification
coefficients = lasso_selector.coef_[0]

# Create a boolean mask where the coefficient is NOT zero
lasso_mask = (coefficients != 0)

X_reduced_columns = X_train_reduced.columns
lasso_features = X_reduced_columns[lasso_mask]

print(f"\n--- Lasso Selection Results ---")
print(f"Original Feature Count (Post-Correlation): {X_train_reduced.shape[1]}")
print(f"Optimal Logistic Regression Feature Count: {len(lasso_features)}")
print(f"Selected Lasso Features: {lasso_features.tolist()}")

In [ ]:
optimal_C = lasso_selector.C_[0]
print(f"Optimal Regularization Strength (C) found by CV: {optimal_C:.4f}")

## 2.2 Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    C=optimal_C,
    penalty='l1',
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
print("Training Final Logistic Regression Model...")
lr_model.fit(X_train_reduced, y_train)

In [ ]:
X_test_reduced = X_test_scaled_df.drop(columns=to_drop)
X_test_reduced

In [ ]:
lr_pred = lr_model.predict(X_test_reduced)
lr_f1 = f1_score(y_test, lr_pred, average='weighted')

print(f"\n--- Logistic Regression Test Performance ---")
print(f"Weighted F1 Score (25 Features): {lr_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=['No Object (0)', 'Foreign Object (1)']))

# 3. Distance Models

## 3.1 Feature Selection

In [ ]:
# Define the SVM Estimator (Linear Kernel is faster and allows feature importance)
svm_estimator = SVC(kernel="linear", random_state=42, probability=True)

svm_rfecv = RFECV(
    estimator=svm_estimator,
    step=1,
    cv=5,
    scoring=f1_scorer,
    min_features_to_select=5,
    n_jobs=-1
)

print("Running RFECV Feature Selection for SVM...")
svm_rfecv.fit(X_train_reduced, y_train)

optimal_k_svm = svm_rfecv.n_features_
print(f"\n--- SVM Feature Selection Results ---")
print(f"Optimal Number of Features for SVM: {optimal_k_svm}")

svm_mask = svm_rfecv.support_
svm_features = X_reduced_columns[svm_mask]
print(f"SVM Selected Features ({optimal_k_svm}): {svm_features.tolist()}")

## 3.2 SVM

In [ ]:
X_train_svm_final = X_train_scaled_df[svm_features]
X_test_svm_final = X_test_scaled_df[svm_features]

svm_estimator = SVC(random_state=42, probability=True)

svm_param_grid = [
    {
        # Tuning for Linear Kernel
        'kernel': ['linear'],
        'C': [0.1, 1, 10],  # Regularization strength
        'class_weight': ['balanced', None]
    },
    {
        # Tuning for RBF Kernel (usually best for non-linear data)
        'kernel': ['rbf'],
        'C': [1, 10, 100],  # Stronger C values often needed for RBF
        'gamma': ['scale', 0.01, 0.1], # Kernel coefficient
        'class_weight': ['balanced', None]
    }
]

svm_grid = GridSearchCV(
    estimator=svm_estimator,
    param_grid=svm_param_grid,
    scoring=f1_scorer,
    cv=5,
    verbose=1,
    n_jobs=-1
)

print("Starting SVM Hyperparameter Tuning...")
start_time = time.time()
svm_grid.fit(X_train_svm_final, y_train)
print(f"SVM Tuning Complete. Time: {round((time.time() - start_time)/60, 2)} min.")

# 4. Get Best Results
best_svm = svm_grid.best_estimator_
print(f"\nBest SVM Params: {svm_grid.best_params_}")
print(f"Best CV F1 Score: {svm_grid.best_score_:.4f}")

## 3.3 K-Nearest Neighbors (KNN)

In [ ]:
knn_estimator = KNeighborsClassifier()

knn_param_grid = {
    'n_neighbors': list(range(1, 16, 2)), # Test odd k values from 1 to 15
    'weights': ['uniform', 'distance'],   # uniform = equal vote; distance = closer points weigh more
    'p': [1, 2]                           # 1=Manhattan distance, 2=Euclidean distance
}

knn_grid = GridSearchCV(
    estimator=knn_estimator,
    param_grid=knn_param_grid,
    scoring=f1_scorer,
    cv=5,
    verbose=1,
    n_jobs=-1
)

print("\nStarting KNN Hyperparameter Tuning...")
start_time = time.time()
knn_grid.fit(X_train_svm_final, y_train)
print(f"KNN Tuning Complete. Time: {round((time.time() - start_time)/60, 2)} min.")

# 4. Get Best Results
best_knn = knn_grid.best_estimator_
print(f"\nBest KNN Params: {knn_grid.best_params_}")
print(f"Best CV F1 Score: {knn_grid.best_score_:.4f}")

In [ ]:
knn_pred = best_knn.predict(X_test_svm_final)
knn_f1 = f1_score(y_test, knn_pred, average='weighted')

print(f"\n--- KNN Test Performance ---")
print(f"Weighted F1 Score: {knn_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, knn_pred, target_names=['No Object (0)', 'Foreign Object (1)']))

# 4. Performance Evaluation

In [ ]:
test_inputs = {
    # Tree/Simple NN models (using 13 RF-selected features)
    'RF': X_test_final,
    'XGBoost': X_test_final,
    'MLP': X_test_final,
    
    # Linear Model (using 25 Lasso-selected features)
    'LR': X_test_reduced,
    
    # Distance Models (using Optimal SVM features)
    'SVM': X_test_svm_final,
    'KNN': X_test_svm_final
}

models = OrderedDict([
    ('RF', best_rf),
    ('XGBoost', best_xgb),
    ('MLP', best_mlp),
    ('LR', lr_model),
    ('SVM', best_svm),
    ('KNN', best_knn)
])

final_results = {}

print("\n--- Final Test Set F1 Score Evaluation ---")
for name, model in models.items():
    X_test_data = test_inputs[name]

    y_pred = model.predict(X_test_data)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    final_results[name] = f1
    print(f"{name:<30}: F1 Score = {f1:.4f}")

# Convert results to a DataFrame for clean presentation
final_results_df = pd.DataFrame(final_results.items(), columns=['Model', 'Weighted F1 Score'])
final_results_df = final_results_df.sort_values(by='Weighted F1 Score', ascending=False)

In [ ]:
print("\n--- FINAL MODEL RANKING (Based on Optimized Performance) ---")
print(final_results_df.to_markdown(index=False))

best_model_name = final_results_df.iloc[0]['Model']
best_model_score = final_results_df.iloc[0]['Weighted F1 Score']

print(f"\n The Best Performing Model is the {best_model_name} with an F1 Score of {best_model_score:.4f}.")

In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"] + plt.rcParams["font.serif"]
plt.rcParams["font.size"] = 10

plt.figure(figsize=(3.5, 3.5)) 

colors = plt.cm.Dark2(np.linspace(0, 1, len(models)))
line_styles = ['-', '--', '-.', ':', (0, (3, 5, 1, 5)), (0, (1, 1))]

for i, (name, model) in enumerate(models.items()):
    X_test_data = test_inputs[name]
    
    if hasattr(model, "predict_proba"):
        y_probs = model.predict_proba(X_test_data)[:, 1]
    else:
        continue 
    
    auc_score = roc_auc_score(y_test, y_probs)
    label_text = f'{name} ({auc_score:.4f})' 
    fpr, tpr, _ = roc_curve(y_test, y_probs)

    plt.plot(fpr, tpr, 
             color=colors[i],
             linestyle=line_styles[i % len(line_styles)], 
             lw=1.2, 
             label=label_text)

plt.plot([0, 1], [0, 1], color='black', lw=0.8, linestyle='--', alpha=0.5, label='Random Chance (0.50)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.legend(loc="lower right", fontsize=7, frameon=True, edgecolor='black', fancybox=False)

plt.grid(True, linestyle=':', alpha=0.4)
plt.gca().set_aspect('equal', adjustable='box') # Keep it perfectly square
plt.tight_layout()

# 7. Save as Vector PDF
plt.savefig('roc_curve_final2.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import cohen_kappa_score, brier_score_loss

kappa_scores = {}
brier_scores = {}

print("\n--- Advanced Performance Metrics ---")

for name, model in models.items():
    X_test_data = test_inputs[name]
    
    # Ensure input is a DataFrame if it was fitted with feature names
    if isinstance(X_test_data, pd.DataFrame):
        X_test_data_input = X_test_data
    else:
        # If it's a NumPy array, it must be the correct array
        X_test_data_input = X_test_data

    # --- 1. Cohen's Kappa ---
    # Requires hard predictions (0 or 1)
    y_pred = model.predict(X_test_data_input)
    kappa = cohen_kappa_score(y_test, y_pred)
    kappa_scores[name] = kappa
    
    # --- 2. Brier Score ---
    # Requires prediction probabilities (for Class 1)
    try:
        # Use predict_proba for Brier Score
        y_probs = model.predict_proba(X_test_data_input)[:, 1]
        brier = brier_score_loss(y_test, y_probs)
        brier_scores[name] = brier
    except AttributeError:
        # Handle models that don't output probabilities easily (like some base SVM setups)
        brier_scores[name] = np.nan
        print(f"Warning: {name} does not support predict_proba for Brier Score.")


# 3. Consolidate and Report
report_df = pd.DataFrame({
    'Model': models.keys(),
    'Cohen_Kappa': list(kappa_scores.values()),
    'Brier_Score': list(brier_scores.values()),
    # Include F1 Score for reference (assuming you have the final_results dict/df)
    'Weighted F1 Score': [final_results.get(k) for k in models.keys()] 
})

report_df = report_df.sort_values(by='Cohen_Kappa', ascending=False)


print("\n--- FINAL METRICS SUMMARY ---")
# Lower Brier Score is better; Higher Kappa and F1 are better.
print(report_df.to_markdown(index=False, floatfmt=".4f"))

In [ ]:
import joblib
from datetime import datetime

best_model_name = final_results_df.iloc[0]['Model']
best_model_score_f1 = final_results_df.iloc[0]['Weighted F1 Score']
best_model = models[best_model_name] # The actual optimized model object

In [ ]:
deployment_dir = 'deployment_artifacts'
if not os.path.exists(deployment_dir):
    os.makedirs(deployment_dir)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f'{best_model_name.replace(" ", "_")}_FOD_Model_{timestamp}.joblib'
scaler_filename = f'FOD_Scaler_{timestamp}.joblib'
features_filename = f'FOD_Features_{timestamp}.txt'

# --- A. Save the Champion Model ---
joblib.dump(best_model, os.path.join(deployment_dir, model_filename))

# --- B. Save the Fitted Scaler ---
# The scaler must be saved because new data must be transformed with the training mean/std.
joblib.dump(scaler, os.path.join(deployment_dir, scaler_filename))

# --- C. Save the Feature List ---
# This list ensures the new data is subsetted correctly.
# Determine which feature list belongs to the best model (RF, Lasso, or SVM)
if best_model_name in ['Random Forest (RF)', 'XGBoost', 'Gradient Boosting (GBT)', 'Neural Network (MLP)']:
    final_feature_list = final_features # Your 13 RF features
elif best_model_name == 'Logistic Regression (LR)':
    final_feature_list = X_reduced_columns # Your 25 Lasso features (post-correlation)
else:
    final_feature_list = svm_features # Your SVM/KNN features

with open(os.path.join(deployment_dir, features_filename), 'w') as f:
    for item in final_feature_list:
        f.write("%s\n" % item)


print("\n✅ Deployment Artifacts Saved Successfully!")
print(f"Model File: {model_filename}")
print(f"Scaler File: {scaler_filename}")
print(f"Feature List: {features_filename}")

# Investigation of the 14 FNs

In [ ]:
y_test_aligned = y_test.reset_index(drop=True)

y_pred_aligned = pd.Series(y_pred)

tp_mask = (y_test_aligned == 1) & (y_pred_aligned == 1)
X_test_TP = X_test_final[tp_mask]

fn_mask = (y_test_aligned == 1) & (y_pred_aligned == 0)
X_test_FN = X_test_final[fn_mask]

tn_mask = (y_test_aligned == 0) & (y_pred_aligned == 0)
X_test_TN = X_test_final[tn_mask]

print(f"Number of False Negatives isolated: {len(X_test_FN)}")

In [ ]:
fn_means = X_test_FN.mean()
tp_means = X_test_TP.mean()
tn_means = X_test_TN.mean()

comparison_df = pd.DataFrame({
    'FN_Mean': fn_means,
    'TP_Mean': tp_means,
    'TN_Mean': tn_means
})

print("\n--- Feature Mean Comparison (FN vs TP vs TN) ---")
print(comparison_df.to_string())

In [ ]:
y_test_aligned = y_test.reset_index(drop=False)
y_pred_series = pd.Series(y_pred, index=y_test_aligned.index)

fn_mask = (y_test_aligned[y_test.name] == 1) & (y_pred_series == 0)
fn_original_indices = y_test.index[fn_mask.values]

print(f"Original Indices of the 14 FNs: {fn_original_indices.tolist()}")

In [ ]:
df_original = pd.read_csv('features_dataset.csv')
df_original.loc[fn_original_indices]